# Day 2 Session 4 -- Demos: Multi-Agent Workflows & Agentic Systems

This notebook contains the **instructor-led demos** for Session 4. Run each cell, observe the output, and follow along as the instructor explains each multi-agent pattern.

**Engineering context:** You are an engineer building multi-agent orchestration systems. Your users (consultants) work in engagement teams where a Partner supervises, Associates specialize by domain, and work flows across functional boundaries. Multi-agent architectures mirror these team dynamics -- supervisor routing, agent handoffs, parallel workstreams, and collaborative deliverable creation.

## Session Goal

Students will learn **5 multi-agent patterns** that mirror real consulting team dynamics:

1. **Supervisor Delegation** -- A Partner routes incoming requests to specialized Associates
2. **Agent Handoffs** -- Strategy team passes context to Operations, then to Implementation
3. **Parallel Workstreams** -- Multiple analysts run due diligence simultaneously, then synthesize
4. **Collaborative Deliverable Creation** -- EM structures, Associate develops, Partner polishes
5. **End-to-End Systems** -- Intent classification + specialist routing + quality review

By the end of this session, you will be able to design and implement any of these patterns using LangGraph's `StateGraph`, conditional edges, and sequential/parallel node composition.

## Setup

In [ ]:
!pip install -q langchain langchain-openai langchain-core langgraph python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from typing import TypedDict, Literal
import json
import os

print("All imports successful!")

---
## Demo 1: Supervisor-Worker Pattern -- Partner Delegates to Associates

The supervisor pattern is the most common multi-agent architecture, and it directly mirrors an engagement team. A **Partner** (supervisor) receives the client request, decides which **Associate** (worker) should handle it based on the nature of the work, and routes accordingly. One Associate handles financial analysis, another develops strategic recommendations, and a third builds implementation roadmaps. The Partner reviews all outputs before they reach the client.

This is the **supervisor-worker pattern** -- a central routing agent delegates to specialized workers and reviews their deliverables.

In [ ]:
# Demo 1a - Define State + Supervisor Node

class SupervisorState(TypedDict):
    task: str
    assigned_to: str
    worker_output: str
    final_response: str

llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def supervisor_node(state: SupervisorState) -> dict:
    """Partner assesses the engagement request and assigns to the right Associate."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner managing an engagement team. "
            "Based on the client request, assign the work to one Associate: "
            "'financial_analyst' (for financial modeling, valuation, or quantitative analysis), "
            "'strategy_consultant' (for market assessment, competitive strategy, or growth recommendations), or "
            "'operations_expert' (for process optimization, implementation planning, or operational improvements). "
            "Respond with just the associate role name."
        )),
        HumanMessage(content=state["task"])
    ])
    assigned = response.content.strip().lower()
    print(f"  Partner assigned to: {assigned}")
    return {"assigned_to": assigned}

print("State and supervisor node defined.")

In [ ]:
# Demo 1b - Define 3 Worker Nodes + Review Node

def financial_analyst_node(state: SupervisorState) -> dict:
    """Associate specializing in financial analysis and modeling."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in financial analysis. "
            "Provide structured, data-driven analysis with key metrics, financial implications, "
            "and quantitative insights. Use consulting frameworks where appropriate."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Financial Analysis Workstream] {response.content}"}

def strategy_consultant_node(state: SupervisorState) -> dict:
    """Associate specializing in strategy and market assessment."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in strategy. "
            "Provide market insights, competitive positioning analysis, and strategic recommendations. "
            "Structure your response with clear hypotheses and supporting evidence."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Strategy Workstream] {response.content}"}

def operations_expert_node(state: SupervisorState) -> dict:
    """Associate specializing in operations and implementation."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate specializing in operations and implementation. "
            "Provide actionable implementation roadmaps, process improvements, and operational recommendations. "
            "Include timelines, resource requirements, and key milestones."
        )),
        HumanMessage(content=state["task"])
    ])
    return {"worker_output": f"[Operations Workstream] {response.content}"}

def review_node(state: SupervisorState) -> dict:
    """Partner reviews the Associate's deliverable before client presentation."""
    response = llm.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner reviewing an Associate's deliverable. "
            "Assess the quality, add executive-level perspective, and finalize the recommendation. "
            "Ensure the output is client-ready with clear, actionable insights."
        )),
        HumanMessage(content=f"Client Request: {state['task']}\n\nAssociate Deliverable:\n{state['worker_output']}")
    ])
    return {"final_response": response.content}

print("Worker nodes and review node defined.")

**Key Concept:** The supervisor pattern mirrors a Partner delegating to Associates. The supervisor sees all requests but does NO analytical work -- it just routes. This separation of concerns makes the system modular. You can add new specialists without changing the supervisor's logic beyond adding a new routing option.

In [ ]:
# Demo 1c - Define Routing + Build Graph

def route_to_worker(state: SupervisorState) -> str:
    a = state["assigned_to"]
    if "financial" in a:
        return "financial_analyst"
    if "strategy" in a:
        return "strategy_consultant"
    return "operations_expert"

graph = StateGraph(SupervisorState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("financial_analyst", financial_analyst_node)
graph.add_node("strategy_consultant", strategy_consultant_node)
graph.add_node("operations_expert", operations_expert_node)
graph.add_node("review", review_node)

graph.set_entry_point("supervisor")
graph.add_conditional_edges("supervisor", route_to_worker, {
    "financial_analyst": "financial_analyst",
    "strategy_consultant": "strategy_consultant",
    "operations_expert": "operations_expert"
})
graph.add_edge("financial_analyst", "review")
graph.add_edge("strategy_consultant", "review")
graph.add_edge("operations_expert", "review")
graph.add_edge("review", END)

app = graph.compile()
print("Graph compiled successfully!")

In [ ]:
# Demo 1d - Run Test Cases

for task in [
    "Assess the financial viability of acquiring a mid-size SaaS company with $50M ARR",
    "Develop a market entry strategy for a luxury automotive brand entering India",
    "Design a lean operating model for a post-merger integration of two regional banks"
]:
    print(f"\nClient Request: {task}")
    result = app.invoke({"task": task, "assigned_to": "", "worker_output": "", "final_response": ""})
    print(f"Partner-Reviewed Deliverable: {result['final_response'][:250]}...")

**Observe:** Each request automatically routes to the right specialist. The Partner review node adds executive perspective regardless of which Associate did the work. The financial request goes to the financial analyst, the market entry request goes to strategy, and the operating model request goes to operations -- all without hardcoded rules, just LLM-based classification.

### Try This

Add a **4th worker** called `digital_specialist` that handles digital transformation, technology adoption, and data & AI strategy requests. You will need to:
1. Add `'digital_specialist'` to the supervisor's routing options in its system prompt
2. Define a `digital_specialist_node` function with appropriate expertise
3. Update the `route_to_worker` function to detect `"digital"` in the assignment
4. Add the node to the graph with edges to `review`
5. Update `add_conditional_edges` to include the new route

Test with: `"Develop a data & AI strategy to personalize customer experiences at scale"`

---
## Demo 2: Agent Handoff -- Strategy to Operations to Implementation

In engagements, work often flows between specialized teams: the **Strategy team** defines the vision and hypotheses, the **Operations team** translates strategy into operational plans, and the **Implementation team** builds the execution roadmap. Each handoff includes structured context -- engagement scope, key findings, and areas requiring attention.

This is the **agent handoff pattern** where one agent transfers control and context to the next agent in a pipeline.

In [ ]:
# Demo 2a - Define State + Strategy Agent

class HandoffState(TypedDict):
    user_request: str
    draft: str
    review_notes: str
    final_output: str
    handoff_context: str

llm_handoff = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def strategy_team_agent(state: HandoffState) -> dict:
    """Strategy team: defines the strategic vision, hypotheses, and recommendations."""
    response = llm_handoff.invoke([
        SystemMessage(content=(
            "You are a McKinsey Strategy team. Analyze the client situation and develop "
            "strategic recommendations. Structure your output with: key hypotheses, "
            "supporting evidence, and top-level strategic recommendations."
        )),
        HumanMessage(content=state["user_request"])
    ])
    context = (
        "Strategy phase complete. Key deliverables: strategic hypotheses validated, "
        "market positioning defined, growth levers identified. "
        "Operations team should focus on translating these into operational plans."
    )
    print(f"  [Strategy Team] Delivered strategic recommendations ({len(response.content)} chars)")
    return {"draft": response.content, "handoff_context": context}

print("HandoffState and strategy agent defined.")

In [ ]:
# Demo 2b - Define Operations + Implementation Agents

def operations_team_agent(state: HandoffState) -> dict:
    """Operations team: translates strategy into operational plans and processes."""
    response = llm_handoff.invoke([
        SystemMessage(content=(
            "You are a McKinsey Operations team. Review the strategy deliverable and "
            "develop an operational plan. Focus on process changes, resource allocation, "
            "organizational structure, and key performance indicators."
        )),
        HumanMessage(content=f"Handoff Context: {state['handoff_context']}\n\nStrategy Deliverable:\n{state['draft']}")
    ])
    print(f"  [Operations Team] Operational plan ready ({response.content[:80]}...)")
    return {"review_notes": response.content}

def implementation_team_agent(state: HandoffState) -> dict:
    """Implementation team: builds the execution roadmap with timelines and milestones."""
    response = llm_handoff.invoke([
        SystemMessage(content=(
            "You are a McKinsey Implementation team. Take the strategy and operational plans "
            "and create a concrete implementation roadmap. Include: phased timeline, "
            "key milestones, resource requirements, risk mitigation, and success metrics."
        )),
        HumanMessage(content=f"Strategy:\n{state['draft']}\n\nOperational Plan:\n{state['review_notes']}")
    ])
    print(f"  [Implementation Team] Roadmap finalized ({len(response.content)} chars)")
    return {"final_output": response.content}

print("Operations and implementation agents defined.")

In [ ]:
# Demo 2c - Build Graph + Run

graph = StateGraph(HandoffState)
graph.add_node("strategy", strategy_team_agent)
graph.add_node("operations", operations_team_agent)
graph.add_node("implementation", implementation_team_agent)
graph.set_entry_point("strategy")
graph.add_edge("strategy", "operations")
graph.add_edge("operations", "implementation")
graph.add_edge("implementation", END)
app_handoff = graph.compile()

result = app_handoff.invoke({
    "user_request": "Our client, a Fortune 500 retailer, needs to develop a digital transformation strategy to compete with e-commerce disruptors",
    "draft": "", "review_notes": "", "final_output": "", "handoff_context": ""
})
print(f"\nImplementation Roadmap:\n{result['final_output'][:400]}...")

**Key Insight:** Each agent handoff includes context from the previous agent. Without `handoff_context`, the operations team wouldn't know what the strategy team concluded. The `handoff_context` field acts like a structured briefing memo -- it tells the next team what was done, what the key findings are, and what they should focus on. In production systems, this context preservation is critical for maintaining coherence across agent boundaries.

---
### QnA Recap -- Demos 1 & 2

Pause here for questions. Key discussion points:

1. **Supervisor vs. Handoff** -- When do you use a central router (Demo 1) vs. a sequential pipeline (Demo 2)?
   - Supervisor: when the task can be fully handled by ONE specialist
   - Handoff: when the task requires MULTIPLE specialists working in sequence, each building on the prior output

2. **Context preservation** -- How much context should you pass between agents?
   - Too little: downstream agents lack information to do good work
   - Too much: token costs increase, context windows fill, agents get confused
   - Best practice: pass a structured summary (like `handoff_context`) plus the full deliverable

3. **Error handling** -- What happens if the supervisor routes incorrectly?
   - In production, add a feedback loop: if the worker detects a mismatch, route back to supervisor
   - LangGraph supports cycles -- the worker can return to the supervisor node

---
## Demo 3: Parallel Execution (Fan-out/Fan-in) -- M&A Due Diligence Workstreams

In an M&A due diligence engagement, multiple workstreams run simultaneously: **financial analysis**, **market research**, and **competitive assessment** all proceed in parallel to meet tight deal timelines. Each workstream produces independent findings, and the Engagement Manager synthesizes them into a unified due diligence report.

This is the **parallel execution pattern** -- independent agents work on different aspects, and a merger step combines the results.

In [ ]:
# Demo 3a - Define State + 3 Parallel Agents

class ParallelState(TypedDict):
    topic: str
    financial_analysis: str
    market_research: str
    competitive_assessment: str
    merged_report: str

llm_parallel = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.5")))

def financial_analysis_agent(state: ParallelState) -> dict:
    """Workstream 1: Financial due diligence -- revenue, margins, projections."""
    r = llm_parallel.invoke([
        SystemMessage(content=(
            "You are a McKinsey financial analyst conducting M&A due diligence. "
            "Analyze the financial aspects: revenue trends, margin structure, "
            "cash flow quality, and valuation considerations. Provide 3 key findings."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Financial Analysis Workstream] Complete")
    return {"financial_analysis": r.content}

def market_research_agent(state: ParallelState) -> dict:
    """Workstream 2: Market due diligence -- TAM, growth drivers, trends."""
    r = llm_parallel.invoke([
        SystemMessage(content=(
            "You are a McKinsey market research analyst conducting M&A due diligence. "
            "Assess the target market: total addressable market (TAM), growth drivers, "
            "regulatory landscape, and market trends. Provide 3 key findings."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Market Research Workstream] Complete")
    return {"market_research": r.content}

def competitive_assessment_agent(state: ParallelState) -> dict:
    """Workstream 3: Competitive due diligence -- positioning, moats, threats."""
    r = llm_parallel.invoke([
        SystemMessage(content=(
            "You are a McKinsey competitive strategy analyst conducting M&A due diligence. "
            "Evaluate competitive dynamics: market positioning, competitive moats, "
            "threat of new entrants, and differentiation. Provide 3 key findings."
        )),
        HumanMessage(content=state["topic"])
    ])
    print("  [Competitive Assessment Workstream] Complete")
    return {"competitive_assessment": r.content}

print("ParallelState and 3 workstream agents defined.")

In [ ]:
# Demo 3b - Define Synthesis Agent + Build Graph + Run

def synthesis_agent(state: ParallelState) -> dict:
    """Engagement Manager synthesizes all workstreams into a unified due diligence report."""
    r = llm_parallel.invoke([
        SystemMessage(content=(
            "You are a McKinsey Engagement Manager synthesizing M&A due diligence findings. "
            "Combine the three workstream outputs into a cohesive executive summary. "
            "Include: overall recommendation (proceed/caution/pass), key risks, "
            "synergy opportunities, and critical next steps."
        )),
        HumanMessage(content=(
            f"Deal Context: {state['topic']}\n\n"
            f"Financial Analysis:\n{state['financial_analysis']}\n\n"
            f"Market Research:\n{state['market_research']}\n\n"
            f"Competitive Assessment:\n{state['competitive_assessment']}"
        ))
    ])
    return {"merged_report": r.content}

graph = StateGraph(ParallelState)
graph.add_node("financial", financial_analysis_agent)
graph.add_node("market", market_research_agent)
graph.add_node("competitive", competitive_assessment_agent)
graph.add_node("synthesis", synthesis_agent)

graph.set_entry_point("financial")
graph.add_edge("financial", "market")
graph.add_edge("market", "competitive")
graph.add_edge("competitive", "synthesis")
graph.add_edge("synthesis", END)

app_parallel = graph.compile()

result = app_parallel.invoke({
    "topic": "Acquisition of a mid-market healthcare technology company specializing in AI-powered diagnostics, with $80M revenue and 25% YoY growth",
    "financial_analysis": "", "market_research": "", "competitive_assessment": "", "merged_report": ""
})
print(f"\nDue Diligence Report:\n{result['merged_report'][:400]}...")

**Gotcha:** LangGraph runs these sequentially (financial -> market -> competitive) because we chained them with `add_edge`. True parallel execution requires `asyncio` or LangGraph's parallel branching with fan-out nodes. But the PATTERN is fan-out/fan-in -- all agents feed into one synthesis step. The key architectural insight is that the agents are **independent** (they don't read each other's output), so they COULD run in parallel. The synthesis agent is the only one that needs all three outputs.

### Try This

Add a **4th parallel workstream** called `regulatory_assessment_agent` that evaluates:
- Regulatory approval requirements (FDA, CE marking, etc.)
- Compliance risks and timeline implications
- Intellectual property landscape

You will need to:
1. Add `regulatory_assessment: str` to `ParallelState`
2. Define the new agent function
3. Add it to the graph between `competitive` and `synthesis`
4. Update `synthesis_agent` to include the regulatory findings in its input

Test with the same healthcare technology acquisition scenario.

---
## Demo 4: Collaborative Deliverable Creation -- Engagement Team Client Presentation

Producing a client deliverable is a collaborative effort. The **Engagement Manager** creates the storyline and structure (using the pyramid principle), an **Associate** develops the detailed content and analyses, and a **Senior Partner** reviews and polishes the final presentation for the Steering Committee.

This is the **collaborative writing pattern** where specialized agents each contribute to building a shared output sequentially.

In [ ]:
# Demo 4a - Define State + EM Storyline

class CollabState(TypedDict):
    topic: str
    outline: str
    draft: str
    polished: str

llm_collab = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0.7")))

def em_storyline(state: CollabState) -> dict:
    """Engagement Manager creates the presentation storyline and structure."""
    r = llm_collab.invoke([
        SystemMessage(content=(
            "You are a McKinsey Engagement Manager creating a presentation storyline. "
            "Develop a clear 3-section structure with the pyramid principle: "
            "start with the answer, then supporting arguments, then evidence. "
            "Include section titles and key messages for each section."
        )),
        HumanMessage(content=f"Create a client presentation storyline for: {state['topic']}")
    ])
    print("  [EM] Storyline created")
    return {"outline": r.content}

print("CollabState and EM storyline node defined.")

In [ ]:
# Demo 4b - Define Associate + Partner Nodes

def associate_content(state: CollabState) -> dict:
    """Associate develops detailed content following the storyline."""
    r = llm_collab.invoke([
        SystemMessage(content=(
            "You are a McKinsey Associate developing content for a client presentation. "
            "Follow the storyline structure and fill in detailed analysis, data points, "
            "and recommendations. Use clear, concise consulting language. "
            "Keep the content executive-ready."
        )),
        HumanMessage(content=f"Engagement Topic: {state['topic']}\n\nStoryline:\n{state['outline']}")
    ])
    print(f"  [Associate] Content developed ({len(r.content)} chars)")
    return {"draft": r.content}

def partner_review(state: CollabState) -> dict:
    """Senior Partner polishes the presentation for Steering Committee readiness."""
    r = llm_collab.invoke([
        SystemMessage(content=(
            "You are a McKinsey Senior Partner reviewing a client presentation. "
            "Polish the content for C-suite readability: sharpen the 'so what', "
            "ensure each section drives toward actionable recommendations, "
            "and add strategic perspective. Make it Steering Committee-ready."
        )),
        HumanMessage(content=state["draft"])
    ])
    print(f"  [Partner] Presentation polished ({len(r.content)} chars)")
    return {"polished": r.content}

print("Associate and Partner nodes defined.")

In [ ]:
# Demo 4c - Build Graph + Run

graph = StateGraph(CollabState)
graph.add_node("em_storyline", em_storyline)
graph.add_node("associate_content", associate_content)
graph.add_node("partner_review", partner_review)
graph.set_entry_point("em_storyline")
graph.add_edge("em_storyline", "associate_content")
graph.add_edge("associate_content", "partner_review")
graph.add_edge("partner_review", END)
app_collab = graph.compile()

result = app_collab.invoke({
    "topic": "Cross-functional transformation program for a global pharmaceutical company entering biosimilars",
    "outline": "", "draft": "", "polished": ""
})
print(f"\nSteering Committee Presentation:\n{result['polished'][:500]}...")

**Observe:** The final output is richer than any single agent could produce. The EM set structure (pyramid principle, 3 sections, clear key messages), the Associate added depth (data points, analysis, specifics), and the Partner added strategic polish (C-suite language, sharpened 'so what', actionable recommendations). Each layer builds on the previous one -- this is the power of collaborative multi-agent workflows.

---
### QnA Recap -- Demos 3 & 4

Pause here for questions. Key discussion points:

1. **Parallel vs. Sequential** -- Demo 3 agents are independent (they don't need each other's output), while Demo 4 agents are dependent (each builds on the previous). How do you decide?
   - If agents need each other's output: sequential (handoff or collaborative)
   - If agents can work independently: parallel (fan-out/fan-in)

2. **When to synthesize** -- In Demo 3, synthesis happens AFTER all workstreams complete. What if one workstream contradicts another?
   - The synthesis agent's job is to reconcile conflicts and present a balanced view
   - In production, you might add a "conflict detection" step before synthesis

3. **Quality in collaboration** -- In Demo 4, what if the Associate drifts from the EM's storyline?
   - The Partner review step catches drift and realigns
   - In production, add explicit checks: does the draft follow the outline's structure?

---
## Demo 5: End-to-End Multi-Agent System -- Engagement Intake & Delivery with Intent Classification

This demo brings together supervisor routing, specialized workers, and quality review into a complete engagement delivery system. A **Triage Partner** classifies incoming client requests by type (strategic question, execution task, or relationship touchpoint), routes to the appropriate specialist, and a **Quality Assurance** step ensures deliverable standards before client delivery.

This mirrors how diverse client interactions are managed through a structured intake and delivery process.

In [ ]:
# Demo 5a - Define State + Intent Classifier

class E2EState(TypedDict):
    user_input: str
    intent: str
    agent_output: str
    quality_score: str
    final_output: str

llm_e2e = ChatOpenAI(model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"), temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0")))

def intent_classifier(state: E2EState) -> dict:
    """Triage Partner classifies the client request type."""
    r = llm_e2e.invoke([
        SystemMessage(content=(
            "You are a McKinsey Partner triaging client requests. "
            "Classify the request as: 'strategic_question' (needs analysis or recommendation), "
            "'execution_task' (needs an action plan or deliverable), or "
            "'relationship' (general check-in or discussion). "
            "Respond with one classification only."
        )),
        HumanMessage(content=state["user_input"])
    ])
    intent = r.content.strip().lower()
    print(f"  Triage: {intent}")
    return {"intent": intent}

print("E2EState and intent classifier defined.")

In [ ]:
# Demo 5b - Define 3 Specialist + Quality Nodes

def strategy_advisor(state: E2EState) -> dict:
    """Handles strategic questions with hypothesis-driven analysis."""
    r = llm_e2e.invoke([
        SystemMessage(content=(
            "You are a McKinsey strategy advisor. Answer the client's strategic question "
            "with structured, hypothesis-driven analysis. Use frameworks like Porter's Five Forces, "
            "value chain analysis, or growth-share matrix as appropriate."
        )),
        HumanMessage(content=state["user_input"])
    ])
    return {"agent_output": r.content}

def execution_lead(state: E2EState) -> dict:
    """Handles execution tasks with actionable implementation plans."""
    r = llm_e2e.invoke([
        SystemMessage(content=(
            "You are a McKinsey execution lead. Develop a step-by-step action plan "
            "for the client's request. Include timeline, resource requirements, "
            "key milestones, and risk factors."
        )),
        HumanMessage(content=state["user_input"])
    ])
    return {"agent_output": r.content}

def relationship_manager(state: E2EState) -> dict:
    """Handles relationship touchpoints with a client-service mindset."""
    r = llm_e2e.invoke([
        SystemMessage(content=(
            "You are a McKinsey relationship manager. Respond to the client with warmth "
            "and professionalism. Acknowledge their situation, provide value-adding insights, "
            "and suggest next steps to deepen the engagement."
        )),
        HumanMessage(content=state["user_input"])
    ])
    return {"agent_output": r.content}

def quality_check(state: E2EState) -> dict:
    """QA check ensures the deliverable meets standards."""
    r = llm_e2e.invoke([
        SystemMessage(content=(
            "You are a McKinsey quality assurance reviewer. Rate this deliverable 1-10 "
            "on: clarity, actionability, and strategic depth. "
            'Output JSON: {"score": N, "feedback": "..."}'
        )),
        HumanMessage(content=f"Client Request: {state['user_input']}\n\nDeliverable: {state['agent_output']}")
    ])
    return {"quality_score": r.content, "final_output": state["agent_output"]}

print("Specialist nodes and quality check defined.")

**Key Insight:** The QA step acts as a production guard -- in a real system, responses scoring below 7 would be flagged for human review before reaching the client. This pattern separates "generation" from "evaluation" -- a fundamental principle in production AI systems. The quality node never generates content; it only judges what the specialist produced.

In [ ]:
# Demo 5c - Build Graph + Run Tests

def route_intent(state: E2EState) -> str:
    if "strategic" in state["intent"] or "question" in state["intent"]:
        return "strategy"
    if "execution" in state["intent"] or "task" in state["intent"]:
        return "execution"
    return "relationship"

graph = StateGraph(E2EState)
graph.add_node("classify", intent_classifier)
graph.add_node("strategy", strategy_advisor)
graph.add_node("execution", execution_lead)
graph.add_node("relationship", relationship_manager)
graph.add_node("quality", quality_check)

graph.set_entry_point("classify")
graph.add_conditional_edges("classify", route_intent, {
    "strategy": "strategy", "execution": "execution", "relationship": "relationship"
})
graph.add_edge("strategy", "quality")
graph.add_edge("execution", "quality")
graph.add_edge("relationship", "quality")
graph.add_edge("quality", END)
app_e2e = graph.compile()

for inp in [
    "What are the key factors driving consolidation in the European banking sector?",
    "Develop a 90-day plan to reduce supply chain costs by 15%",
    "We enjoyed the last workshop -- can we schedule a follow-up to discuss next steps?"
]:
    print(f"\nClient: {inp}")
    r = app_e2e.invoke({"user_input": inp, "intent": "", "agent_output": "", "quality_score": "", "final_output": ""})
    print(f"Deliverable: {r['final_output'][:200]}...")
    print(f"QA Score: {r['quality_score'][:100]}")

---
### QnA Recap -- Demo 5

Pause here for questions. Key discussion points:

1. **Intent classification robustness** -- What if a request is ambiguous (e.g., "Help us think about whether to restructure our supply chain")?
   - Could be strategic (should we?) or execution (how do we?)
   - In production, add a "clarification" intent that asks the client to specify
   - Or add confidence thresholds: if the classifier is < 70% confident, escalate to a human

2. **Quality gate decisions** -- What should happen when quality score is low?
   - Option A: Regenerate (send back to the specialist with the QA feedback)
   - Option B: Escalate (flag for human review)
   - Option C: Augment (pass to a senior specialist to improve)
   - LangGraph supports all three via conditional edges from the quality node

3. **Combining patterns** -- Demo 5 combines supervisor routing (intent classifier) with quality review. How would you add handoffs?
   - After the specialist generates output, add a "refinement" agent before QA
   - Or chain specialists: strategy_advisor -> execution_lead for requests that need both

---
## Summary

This demo notebook covered five multi-agent patterns:

| # | Pattern | Key Mechanism | When to Use |
|---|---------|--------------|-------------|
| 1 | **Supervisor-Worker** | Conditional edges from central router | Task can be handled by ONE specialist |
| 2 | **Agent Handoff** | Sequential edges with context passing | Task requires MULTIPLE specialists in sequence |
| 3 | **Parallel Execution** | Fan-out/fan-in with synthesis | Independent subtasks that feed into one summary |
| 4 | **Collaborative Deliverable** | Sequential build-up of shared output | Multiple roles contribute to ONE document |
| 5 | **End-to-End System** | Classification + routing + quality gate | Production system with intake, processing, and QA |

**Next:** In the exercises notebook, you will build these patterns yourself, starting with a supervisor-worker system and progressing to a full engagement delivery pipeline.